**This part is required for all the notebook**

In [ ]:
import sys
sys.path.append("lib")

from main import Notebook, HandsOn3CatsnifferUI

nb = Notebook()

# Notes:
# Fix typing error:
# pip install --upgrade typing_extensions
# !pip install 

---

# Hands-on Lab #3: Catsniffer + Minino = BLE Unsecured Connections

## Objectives
- Understand BLE pairing vs unpaired (unauthenticated) interactions
- Use CatSniffer to detect GATT write operations
- Control devices (e.g., LED boards) using Minino for BLE read/write operations.


## Challenge
Capture BLE connection attempts to the lab's LED devices. Identify the unencrypted GATT writes being sent. Replay or forge your write operations to change LED states without prior pairing.

## Walk-Through
First of all we need to flash our RP2040 to control the CC1352P7 with the firmware `SerialPassthroughwithboot_RP2040_v1.1.uf2`

![Catsniffer Components Diagram](static/block_diagrams.png)

### Flash firmware

> Run the next code, and after that follow the steps to put the bootloader mode


In [ ]:
serialpass_file = "serialpassthroughwithboot_rp2040_v1.1.uf2"
nb.flash_uf2_firmware(serialpass_file)


### RP2040 bootloader mode:

![Catsniffer Board](static/banner.jpg)


- Press and hold boot RP2040
- While Holding Boot RP2040, press Reset RP2040
- The device will be shown as a thumbdrive

![RP 2040 Boot](static/RP_USB.png)


### Flashing CC1352:

For this lab, we will use the ` sniffle_cc1352p7_1m.hex`, this is a Multiprotocol sniffer from Texas Instrument.

In [ ]:
sniffer_file = "sniffle_cc1352p7_1M.hex"
nb.flash_cc_fiwmare(sniffer_file)

**Note**: A *successful* upload will produce an output similar to:
```bash
Firmware uploaded successfully.

[SUCCESS] Firmware loaded successfully.
```

If you see anything different, please contact the trainer.

### Understanding Minino
Minino has a built-in CLI for BT operations; you need to open the serial terminal to access the Minino terminal. Turn on the minino and connect it to the computer via USB. Now, execute the following command to open the terminal:

```bash
picocom -b 115200 SERIAL_PORT
```

In order to know all the BT-related commands, you can type: `help BT`
```bash
minino > Type 'help <command>' to get help for a specific command
  gattcmd_scan
  gattcmd_enum
  gattcmd_write
  gattcmd_recon
```

In this case, we will first be using the command scan `gattcmd_scan`, so we can get more information about it using the `help` command: `help gattcmd_scan`.


#### Our target
We're looking for **TAPE LIGHTS** ADV packet
```bash
|       00:00:00:9a:19:55 |    -60      |<- TAPE LIGHTS
```

So when have the device address, we need to inspect *GATT* services and characteristics, which can be done with the command: `gattcmd_recon`


### Minino Terminal

In [ ]:
la32ui = HandsOn3CatsnifferUI()
la32ui.display_ui_minino_terminal(True)


The `gattcmd_recon` command will scan all characteristics every 30 seconds:
> If you are using a terminal application like picocom, you can stop the scan at any time by pressing `CTRL+C` once you have collected the necessary information.

Now that we know where the lights expect a **write** operation, we could experiment by sending random data to the device to observe any color changes or malfunctions.
However, since we have both the **original control application** and the **CatSniffer** device, we can attempt a more precise approach: capturing the **exact data** sent to the device.
To do this, we will use the **Sniffle** project, which allows us to sniff across all three BLE advertising channels and follow active connections.


### Catsniffer with Sniffle

**Note**: You will need to update the CatSniffer firmware using the Catnip tool in order to load the Sniffle firmware before continuing. If you don’t have the firmware yet, please return to the [Flash firmware](#flash-firmware) and [Flashing CC1352](#flashing-cc1352) sections of this document.

#### Open wireshark:

In [ ]:
la32ui.display_open_wireshark_btn()

#### Configure Sniffle Extcap

We need to configure our extcap, and for that we will use the information obtained with `Minino`:
- **MAC**: 00:00:00:9a:19:55

When you open Wireshark, it will display the available capture interfaces. Locate the `Sniffle BLE sniffer: sniffle` interface and double-click the configuration icon ![Extcap icon](static/extcap_icon.png) at the beginning of the interface name. 


When we open *wireshark* will display the captures interfaces, search the `Sniffle BLE sniffer: sniffle` interface and double click on the ! at the start of the name of the interface.

![Sniffle Extcap Configuration](static/ws_sniffle.png)

This will open a new modal window where you must configure the following options:

- Sniffer serial port: Select the CatSniffer serial port
- Sniffer mode: Connection Following
- Advertising channel: Auto
- Minimum RSSI: -127
- MAC Address: 00:00:00:9a:19:55

> As a reference, the CatSniffer board will appear with “- Arduino” appended to the serial port path.


![Sniffle Extcap Configuration](static/sniffle_config.png)


When the device is in advertising mode, the sniffer will display information similar to the example below:

![Sniffle Example Capture](static/sniffle_capture.png)

**Next, use the application to connect to the device**. When the connection is established, you should see the handshake process appear in the sniffer output.

![Sniffle Connection](static/sniffle_connection.png)

After the handshake is complete, continue monitoring the communication stream. This will allow you to capture the specific data packets sent to the relevant characteristic.

![Sniffle Packet](static/sniffle_packet.png)

![Sniffle Packet Details](static/sniffle_value.png)

**At this stage, we can identify and leverage Minino’s features to send different payloads of the same length, in order to observe varying device behaviors.**

In Wireshark, we identified the Handle: `0x000e`. Earlier, when running the `recon` command on the Minino, we obtained the same handle value (`000e`), which corresponds to the **characteristic value**: `fff3`.

We will use this characteristic value for our write operations.

```bash

gattcmd_write 00:00:00:9a:19:55 fff3 a4b517db0f8c46d82b7d85af9b2e5e923e

gattcmd_write 00:00:00:9a:19:55 fff3 7f5eb2f81356610709d03b60219079bc8b

```


**Before running the next terminal command, make sure you don’t have an active connection open in the previously used terminal window.**

In [ ]:
la32ui.display_ui_minino_terminal()

This example demonstrates a typical sniffing scenario involving an unencrypted connection.